> 在线讲义：[第 6 章 · 策略 policy π(a|s)：智能体的"行事手册"](https://wikieden.github.io/rl-from-scratch/lessons/06-policy.html) · 本书《从零学强化学习》

# 第 6 章 · 策略 policy π(a|s)：智能体的"行事手册"

本章回答：智能体是怎么选动作的？"策略"写出来长什么样？为什么策略常常是随机的？

配套讲义：仓库里 `cs285-deep-rl/notes/lessons/06-policy.md`（讲解、动画、练习都在讲义里）。

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU。

In [ ]:
import numpy as np

np.set_printoptions(precision=3, suppress=True)  # 打印保留 3 位小数
print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 代码 | 跑出来 |
|---|---|---|
| 策略表 | `ROBOT_PI = {0: {0: 0.9, 1: 0.1}, 1: {0: 0.2, 1: 0.3, 2: 0.5}}` | — |
| 掷 1000 次 | `sample_action(pi, 1, rng)` | 搜索 193、等待 280、充电 527 |
| 一步平均奖励 | `expected_immediate_reward(P, pi, s)` | 高电 1.9，低电 0.4 |

## 1. 先把要用的东西定义好

这几格只是定义（环境、函数），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**常量 `ROBOT_ACTIONS`**（来自 `rl_envs.py`）

In [ ]:
ROBOT_ACTIONS = ["搜索", "等待", "充电"]

**常量 `ROBOT_PI`**（来自 `rl_envs.py`）

In [ ]:
ROBOT_PI = {0: {0: 0.9, 1: 0.1}, 1: {0: 0.2, 1: 0.3, 2: 0.5}}

**常量 `ROBOT_STATES`**（来自 `rl_envs.py`）

In [ ]:
ROBOT_STATES = ["高电", "低电"]

**函数 `expected_reward`**（来自 `ch05_mdp.py`）：下面要用到的小工具。

In [ ]:
def expected_reward(P, s, a):
    return sum(p * r for p, _, r, _ in P[s][a])

**函数 `expected_immediate_reward`**（来自 `ch06_policy.py`）：E_π[R_{t+1} | S_t = s] = Σ_a π(a|s) r(s, a)。

In [ ]:
def expected_immediate_reward(P, pi, s):
    """E_π[R_{t+1} | S_t = s] = Σ_a π(a|s) r(s, a)。"""
    return sum(pi[s][a] * expected_reward(P, s, a) for a in pi[s])

**常量 `ALPHA`, `BETA`**（来自 `rl_envs.py`）

In [ ]:
ALPHA, BETA = 0.8, 0.7

**常量 `R_SEARCH`, `R_WAIT`, `R_RESCUE`**（来自 `rl_envs.py`）

In [ ]:
R_SEARCH, R_WAIT, R_RESCUE = 2.0, 1.0, -3.0

**函数 `robot_mdp`**（来自 `rl_envs.py`）：返回 P[s][a]；状态 0 = 高电、1 = 低电；动作 0 = 搜索、1 = 等待、2 = 充电（只有低电能充电）。

In [ ]:
def robot_mdp():
    """返回 P[s][a]；状态 0 = 高电、1 = 低电；动作 0 = 搜索、1 = 等待、2 = 充电（只有低电能充电）。"""
    hi, lo = 0, 1
    P = {
        hi: {
            0: [(ALPHA, hi, R_SEARCH, False), (1 - ALPHA, lo, R_SEARCH, False)],
            1: [(1.0, hi, R_WAIT, False)],
        },
        lo: {
            0: [(BETA, lo, R_SEARCH, False), (1 - BETA, hi, R_RESCUE, False)],
            1: [(1.0, lo, R_WAIT, False)],
            2: [(1.0, hi, 0.0, False)],
        },
    }
    return P

**函数 `sample_action`**（来自 `ch06_policy.py`）：下面要用到的小工具。

In [ ]:
def sample_action(pi, s, rng):
    acts = list(pi[s])
    return acts[rng.choice(len(acts), p=[pi[s][a] for a in acts])]

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
P, pi, rng = robot_mdp(), ROBOT_PI, np.random.default_rng(0)

In [ ]:
for s in pi:
        print(ROBOT_STATES[s], {ROBOT_ACTIONS[a]: p for a, p in pi[s].items()})

In [ ]:
picks = [sample_action(pi, 1, rng) for _ in range(1000)]

In [ ]:
print("低电时抽 1000 次：", {ROBOT_ACTIONS[a]: picks.count(a) for a in pi[1]})

In [ ]:
for s in pi:
        print(f"E[下一步奖励 | {ROBOT_STATES[s]}] =", round(expected_immediate_reward(P, pi, s), 4))

## 3. 代码里才看得到的

`rng.choice(n, p=...)` 就是"加权骰子"。神经网络策略（第 20 章）里同样的事写成 `torch.distributions.Categorical(probs).sample()`。

## 4. 自己改一改

讲义里建议的几个小实验（数字是实跑出来的，改完对一对）：

1. `sample_action(pi, 1, rng)` 只掷 10 次，换几个种子：每次数出来的比例都和 0.2 / 0.3 / 0.5 差得很远；掷 10000 次时就很接近了。
2. 把 `ROBOT_PI` 里低电的策略改成搜索 0.6、等待 0.2、充电 0.2，再算 `expected_immediate_reward(P, pi, 1)`，看平均奖励是变大还是变小，想一想为什么。

讲义末尾的练习题也可以在下面的空格子里用代码验算：先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 策略是手册：每个状态各动作选几成，每行加起来是一；
> 随机策略掷骰子，确定策略只一招；
> 一步平均：先 π 后 p，两层加权；
> 跑样本、评估、改策略，所有算法都在转这个圈。